In [1]:
from typing import *
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql import functions as fn
from functools import reduce
from io import *
import os
import glob
import json

dateColumnName = 'date'


# source_dir = "/data/geolocation-data/raw/cell_traces/bogor/2025_10_10"
source_dir = "/data/geolocation-data/raw/cell_traces/bogor/"
source_dir_additional = "/storage/geolocation-data/raw/cell_traces/bogor/"
source_files = glob.glob(source_dir + '**/*.csv', recursive=True)
source_files_additional = glob.glob(source_dir_additional + '**/*.csv', recursive=True)

target_dir = '/data/geolocation-data/processed/cell_traces'
output_cleansing_dir = f'{target_dir}/cleansing'
output_normalization_dir = f'{target_dir}/normalization'
output_aggregation_dir = f'{target_dir}/aggregation'

spark = SparkSession.builder \
    .config("spark.driver.memory", "32g") \
    .config("spark.memory.fraction", "0.8") \
    .appName("CellTraces") \
    .master("local[*]") \
    .getOrCreate()


spark.sparkContext.setLogLevel("INFO")

spark.conf.set("spark.sql.debug.maxToStringFields", 10000)
spark.conf.set("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
spark.conf.set("mapreduce.input.fileinputformat.input.dir.recursive", "true")
spark.conf.set("fs.globstatus.glob.recursive", "true")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/15 20:02:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/15 20:02:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


<h3>Cell Traces Summary</h3>


In [ ]:
from pyspark.sql import functions as fn

df_dummy: Optional[DataFrame] = None
df_null: Optional[DataFrame] = None
df_summary: Optional[DataFrame] = None
dirPath: Optional[str] = None
listOfDF: Optional[List[DataFrame]] = []

df_dummy = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "\\N")             # treat \N as null
    .option("nanValue", "NaN")           # treat "NaN" as actual NaN
    .csv(source_files)
).replace([" ", "", "NULL", "nan", "NaN"], None)

total_rows = df_dummy.count()

if (total_rows > 0):
    print(f"Rows : {total_rows} \n\n")
    # df_initial.show(100, truncate=False)

    exprs = [
        fn.format_number(
            (fn.sum(fn.when(fn.col(c).isNull(), 1).otherwise(0)) / fn.lit(total_rows) * 100), 2).alias(c)
        for c in df_dummy.columns
    ]

    df_null = df_dummy.agg(*exprs)

    n = len(df_dummy.columns)
    stack_expr = "stack({0}, {1}) as (Column_Name, Null_Percentage)".format(
        n,
        ", ".join([f"'{c}', `{c}`" for c in df_dummy.columns])
    )

    df_summary = (
        df_null
        .selectExpr(stack_expr)
        .orderBy(fn.col("Null_Percentage").desc())
    )

    # df_summary.show(500, truncate=False)
    df_summary.write.option("header", True).csv(f"{target_dir}/summary")

<h3>Cell_traces Initial Data Cleansing</h3>


In [ ]:
df_initial: Optional[DataFrame] = None
df_cleansing: Optional[DataFrame] = None
df_dummy: Optional[DataFrame] = None
dirPath: Optional[str] = None
listOfDF: Optional[List[DataFrame]] = []
listOfDropColumn: Optional[List[str]] = [
    'GUTI', 'TMSI', 'P_TMSI', 'IRAT_TO_UMTS_ATTEMPT', 'IRAT_TO_GERAN_ATTEMPT',
    'IRAT_TO_CDMA_ATTEMPT', 'IRAT_FROM_UMTS_ATTEMPT', 'IRAT_FROM_GERAN_ATTEMPT',
    'IRAT_FROM_CDMA_ATTEMPT', 'RRC_FAILURE_CAUSE_LAST', 'NAS_ESM_FAILURE_LAST',
    'RADIO_BEARER_TYPE', 'LATENCY_DRB_DL_AVERAGE', 'IRAT_TO_UMTS_FAILURE',
    'IRAT_TO_GERAN_FAILURE', 'IRAT_TO_CDMA_FAILURE', 'IRAT_FROM_UMTS_FAILURE',
    'IRAT_FROM_GERAN_FAILURE', 'IRAT_FROM_CDMA_FAILURE', 'BLIND_REDIR_4G_3G',
    'VOLTE_STATUS', 'ACCESS_POINT_NAME', 'SRVCC_TYPE', 'APN', 'LATENCY_DRB_DL_SAMPLE',
    'IMSI_GROUP_ID', 'VENDOR_DROP_CAUSE', 'UL_MBR', 'DL_MBR', 'UE_CATEGORY_NB',
    'LPP_ENABLE', 'VOLTE_ENABLE', 'SRS_BASED_ANTE_SWITCH_ENABLE', 'DCNR', 'POS_LAST_RSCP',
    'POS_LAST_ECN0', 'POS_LAST_RSSI', 'IRAT_TO_UMTS_SUCCESS', 'IRAT_TO_GERAN_SUCCESS',
    'HO_4G_LOC', 'POS_INDOOR_GAIN', 'X2AP_FAILURE_CAUSE_LAST', 'RRC_FAILURE_CNT',
    'RRC_FAILURE_LAST', 'S1_HO_INTRA_FAILURE', 'S1_HO_INTER_FAILURE', 'MDT_FLAG',
    'X2AP_FAILURE_CNT', 'X2AP_FAILURE_LAST', 'S1AP_FAILURE_CNT', 'S1AP_FAILURE_LAST',
    'S1AP_FAILURE_CAUSE_LAST', 'S1_HO_INTER_ATTEMPT', 'BLIND_REDIR_4G_2G',
    'CSFB_LTE_SETUP_TIME', 'HO_CONTAINER_OUT', 'S1_HO_INTRA_ATTEMPT',
    'X2_HO_INTER_FAILURE', 'X2_HO_INTRA_FAILURE', 'RRC_RE_ESTB_CAUSE',
    'POS_FIRST_BLDG_ID', 'POS_LAST_BLDG_ID', 'MOVING_SPEED', 'MOVING_TYPE',
    'HO_CONTAINER_IN', 'POS_FIRST_OVERSHOOT_DISTANCE', 'INTER_ENB_INTER_FREQ_ATTOUT',
    'INTER_ENB_INTRA_FREQ_ATTOUT', 'X2_HO_INTRA_ATTEMPT', 'LAST_MME_ID',
    'X2_HO_INTER_ATTEMPT', 'START_SINR_PUCCH', 'END_SINR_PUCCH', 'DATA_COMPRESS_UL_ENABLE',
    'POS_INDOOR_CONFIDENCE', 'VO_WIFI_ENABLE', 'CA_INDICATOR_EARFCN', 'CA_INDICATOR_FREQ_BAND',
    'CA_INDICATOR', 'MIMO_CAP', 'UE_LTE_CATEGORY_UTRA', 'UE_LTE_CATEGORY_GERAN',
    'CSFB_2G_WITH_SI_ENABLE', 'UE_LTE_CATEGORY', 'VOLTE_CAP', 'SRVCC_CAP', 'MME_SGW_IP_ADD_1',
    'MME_SGW_TEID_1', 'MME_SGW_IP_ADD_2', 'MME_SGW_TEID_2', 'MAKE_ID', 'MODEL_ID', 'MIMO_INDICATOR',
    'POS_FIRST_TILE_LON', 'POS_FIRST_TILE_LAT', 'POS_LAST_TILE_LON', 'POS_LAST_TILE_LAT',
    'CSFB_IND', 'NR_FLAG', 'LTE_CAT', 'CA_BAND_COM_MAX', 'CA_BAND_COM_ALL', 'CA_BW_CLASS_UL',
    'CA_MIMO_CAP_UL', 'CA_BW_CLASS_DL', 'CA_MIMO_CAP_DL', 'UE_CATEGORY_UL', 'UE_CATEGORY_DL',
    'UE_CAP_CONTAINER_5', 'UE_CAP_CONTAINER_4_5', 'UE_CAP_CONTAINER_4', 'UE_CAP_CONTAINER_3',
    'IMSI_IMEI_METHOD', 'POS_LAST_TILE', 'POS_LAST_SUB_REGION', 'POS_LAST_S_SUB_REGION',
    'POS_FIRST_TILE', 'POS_FIRST_SUB_REGION', 'POS_FIRST_S_SUB_REGION', 'POS_FIRST_S_SIMULATION',
    'POS_FIRST_REG_ID', 'POS_LAST_REG_ID', 'POS_FIRST_ROAD', 'POS_LAST_ROAD',
    'INITIAL_MME_UE_S1AP_ID', 'INITIAL_ENB_UE_S1AP_ID', 'INITIAL_MME_ID',
    'LAST_MME_UE_S1AP_ID', 'ERAB_ID', 'QCI_ID', 'REF_ID', 'POS_LAST_LON', 'POS_LAST_LAT',
    'POS_LAST_ALTITUDE', 'POS_FIRST_LON', 'POS_FIRST_LAT', 'POS_FIRST_ALTITUDE', 'POS_LAST_LON',
    'POS_LAST_LAT', 'POS_LAST_ALTITUDE', 'POS_FIRST_LON', 'POS_FIRST_LAT', 'POS_FIRST_ALTITUDE',
    'ERAB_SETUP_FAILURE', 'NAS_EMM_FAILURE_LAST', 'START_CQI', 'END_CQI', 'LATENCY_RRC_ESTABLISHMENT',
    'LATENCY_ERAB_ESTABLISHMENT', 'A_GPS_SUPPORT', 'EUTRA_AS_REL', 'FGI_FDD', 'FGI_TDD', 'NR_RF_BAND',
    'RF_BAND_NB', 'POWER_CLASS_BAND', 'IRAT_UTRA_TDD', 'UE_POS_METHOD', 'COMP_DL_ENABLE', 'ROHC_ENABLE',
    'MO_VOICE_RRC_REQ_ENABLE', 'VOICE_PREFERENCE', 'PU_ID', 'POS_LAST_LOC', 'POS_LAST_EUTRABAND',
    'POS_LAST_EARFCN', 'POS_LAST_S_EUTRABAND', 'POS_LAST_S_EARFCN', 'POS_LAST_S_ENODEB', 'POS_FIRST_LOC',
    'POS_FIRST_EUTRABAND', 'POS_FIRST_EARFCN', 'POS_FIRST_S_EUTRABAND', 'POS_FIRST_S_EARFCN'
]

imsi_column_name = 'IMSI'

# /data/geolocation-data/raw/cell_traces/bogor/
#   ---- 2010_10_10/
#   ---- 2010_10_11/
#   ---- 2010_10_12/
#   ---- 2010_10_13/
#   ---- 2010_10_14/


df_initial = (
    spark.read
    .option("recursiveFileLookup", "true")
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "\\N")  # treat \N as null
    .option("nanValue", "NaN")   # treat "NaN" as actual NaN
    .option("mode", "PERMISSIVE")
    # .csv(source_dir)
    .csv(source_dir_additional)
).replace([" ", "", "NULL", "nan", "NaN"], None)


# CAST IMSI as string
df_initial = df_initial.withColumn(imsi_column_name, fn.col(imsi_column_name).cast("string"))

total_rows = df_initial.count()

if (total_rows > 0):
    print(f"Rows of raw data : {total_rows} \n\n")
    df_initial.show(10, truncate=False)


# recursively read the file based on directory path, because we need to add new column based on the path (date)
# for file in source_files:
#     dirPath = os.path.basename(os.path.dirname(file))

#     df_dummy = (
#         spark.read
#         .option("header", True)
#         .option("inferSchema", True)
#         .option("nullValue", "\\N")  # treat \N as null
#         .option("nanValue", "NaN")   # treat "NaN" as actual NaN
#         .option("mode", "PERMISSIVE")
#         .csv(file)
#     ).replace([" ", "", "NULL", "nan", "NaN"], None)

#     # Add new column name based on directory path
#     df_dummy = df_dummy.withColumn(dateColumnName, to_date(lit(dirPath), 'yyyy_MM_dd'))
#     listOfDF.append(df_dummy)


# # # concat the list into new DataFrame
# df_initial = reduce(
#     lambda df1, df2: df1.unionByName(df2),
#     listOfDF
# )


# drop column define in the list
df_cleansing = df_initial.drop(*listOfDropColumn)

total_rows = df_cleansing.count()

if (total_rows > 0):
    print(f"Rows of cleanse data : {total_rows} \n\n")
    df_cleansing.show(10, truncate=False)
    # df_cleansing.write.partitionBy("DATA_DATE").mode("overwrite").option("header", True).parquet(f'{output_cleansing_dir}/parquet')
    df_cleansing.write.partitionBy("DATA_DATE").mode("append").option("header", True).parquet(f'{output_cleansing_dir}/parquet')

In [ ]:
# total_rows = df_initial.count()

# if (total_rows > 0):
#     print(f"Rows of raw data : {total_rows} \n\n")
#     df_initial.show(1000, truncate=False)

<h3>Data Preparation & Normalization</h3>


In [3]:
# from pyspark.sql import functions as fn

df_source: Optional[DataFrame] = None
df_filter: Optional[DataFrame] = None


listOfNumericColumn: Optional[List[str]] = [
    'DURATION', 'UL_VOLUME', 'DL_VOLUME',
    'UL_THROUPUT', 'DL_THROUPUT', 'UL_THROUPUT_MAX',
    'DL_THROUPUT_MAX', 'AVG_RSRP', 'AVG_RSRQ',
    'POS_FIRST_RSRP', 'POS_LAST_RSRP', 'FIRST_TA',
    'LAST_TA', 'AVG_TA', 'PING_PONG_HO_CNT',
    'START_SINR_PUSCH', 'END_SINR_PUSCH',
    'POS_FIRST_S_RSRP', 'POS_FIRST_S_RSRQ'
]

listOfIdColumns: Optional[List[str]] = [
    'START_CELL_ID', 'END_CELL_ID',
    'CALL_ID',
]

listOfKPIColumns = [
    "DURATION", "UL_VOLUME", "DL_VOLUME",
    "UL_THROUPUT", "DL_THROUPUT",
    "UL_THROUPUT_MAX", "DL_THROUPUT_MAX",
    "AVG_RSRP", "AVG_RSRQ",
    "POS_FIRST_RSRP", "POS_LAST_RSRP",
    "POS_FIRST_S_RSRP", "POS_FIRST_S_RSRQ",
    "START_SINR_PUSCH", "END_SINR_PUSCH",
    "FIRST_TA", "LAST_TA", "AVG_TA",
    "PING_PONG_HO_CNT"
]

start_cell_id_column_name = 'START_CELL_ID'
end_cell_id_column_name = 'END_CELL_ID'
src_cell_id_column_name = 'SRC_CELL_ID'
dst_cell_id_column_name = 'DST_CELL_ID'
avg_rsrp_column_name = 'AVG_RSRP'
avg_rsrq_column_name = 'AVG_RSRQ'
avg_ta_column_name = 'AVG_TA'
start_sinr_column_name = 'START_SINR_PUSCH'
end_sinr_column_name = 'END_SINR_PUSCH'
call_type_column_name = 'CALL_TYPE'
call_status_column_name = 'CALL_STATUS'
start_enodeb_column_name = 'START_ENODEB_ID'
end_enodeb_column_name = 'END_ENODEB_ID'
pos_first_cell_column_name = 'POS_FIRST_CELL'
pos_last_cell_column_name = 'POS_LAST_CELL'
start_enodeb_column_name = 'POS_FIRST_ENODEB'
pos_last_enodeb_column_name = 'POS_LAST_ENODEB'


# df_filter = df_cleansing.select('*')

df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "\\N")             # treat \N as null
    .option("nanValue", "NaN")              # treat "NaN" as actual NaN
    .parquet(f'{output_cleansing_dir}/parquet')
).replace([" ", "", "NULL", "nan", "NaN"], None)


df_filter = df_source.select('*')


# Standardize the column type
# ============================================================================
# cast the columns to double
for column in listOfNumericColumn:
    df_filter = df_filter.withColumn(column, fn.col(column).cast("double"))

# cast the columns to long
for column in listOfIdColumns:
    df_filter = df_filter.withColumn(column, fn.col(column).cast("long"))

# ============================================================================


# Filter the Data
# ==================================================================================================================================


# filter / keep the row if any KPI columns is not null
condition = None

for column in listOfKPIColumns:
    expression = fn.col(column).isNotNull()     # this will return spark sql expression
    condition = expression if condition is None else (condition | expression)   # return expression otherwise return concat value


df_filter = df_filter.filter(
    (fn.col('START_CELL_ID').isNotNull() & fn.col('END_CELL_ID').isNotNull()) | condition
)


# Standard value by 3GPP
# ===============================================
# RSRP:    -140 to -40
# RSRQ:    -20 to 0
# SINR:    -10 to 40
# TA:      0 to 128   ( 1 = approx 78m )
# ===============================================

# filter the KPI based on above condition

df_filter = df_filter.filter(
    # RSRP
    ((fn.col(avg_rsrp_column_name) >= -140) & (fn.col(avg_rsrp_column_name) <= -40)) | fn.col(avg_rsrp_column_name).isNull()
).filter(
    # RSRQ
    ((fn.col(avg_rsrq_column_name) >= -20) & (fn.col(avg_rsrq_column_name) <= 0)) | fn.col(avg_rsrq_column_name).isNull()
).filter(
    # START SINR
    ((fn.col(start_sinr_column_name) >= -10) & (fn.col(start_sinr_column_name) <= 40)) | fn.col(start_sinr_column_name).isNull()
).filter(
    # END SINR
    ((fn.col(end_sinr_column_name) >= -10) & (fn.col(end_sinr_column_name) <= 40)) | fn.col(end_sinr_column_name).isNull()
).filter(
    # TA
    ((fn.col(avg_ta_column_name) >= 0) & (fn.col(avg_ta_column_name) <= 100)) | fn.col(avg_ta_column_name).isNull()
)


# CALL_TYPE:
# 21: Data Call
# 22: Signalling Call
# 23: VoLTE
# 24: SMS
# 29: Unspecified Call

# CALL_STATUS:
# 1: Normal Call
# 2: Drop Call
# 3: Block Call
# 4: Unspecified Call
# 5: CS-Fallback Call
# 6: Setup Failure Call
# 7; X2 HO Out (Incomplete Correlation)

# filter only if the below value != -1     (-1 is for simulation done by CovMo)
df_filter = df_filter.filter(
    fn.col(start_enodeb_column_name) != -1
).filter(
    fn.col(end_enodeb_column_name) != -1
).filter(
    fn.col(pos_first_cell_column_name) != -1
).filter(
    fn.col(pos_last_cell_column_name) != -1
).filter(
    fn.col(start_enodeb_column_name) != -1
).filter(
    fn.col(pos_last_enodeb_column_name) != -1
).filter(
    # filter only CALL_TYPE 21,23
    fn.col(call_type_column_name).isin(21, 23)
).filter(
    # filter only CALL_STATUS: 1,2,3,5,6,
    fn.col(call_status_column_name).isin(1, 2, 3, 5, 6)
)


df_filter = (
    df_filter
    .withColumn(
        src_cell_id_column_name, fn.concat_ws(
            '_', fn.col(start_enodeb_column_name).cast('string'), fn.col(start_cell_id_column_name).cast('string')
        )
    ).withColumn(
        dst_cell_id_column_name, fn.concat_ws(
            '_', fn.col(end_enodeb_column_name).cast('string'), fn.col(end_cell_id_column_name).cast('string')
        )
    )
)

# ==================================================================================================================================

total_rows = df_filter.count()

if (total_rows > 0):
    print(f"Rows : {total_rows} \n\n")
    df_filter.show(10, truncate=False)
    df_filter.write.partitionBy("DATA_DATE").mode("overwrite").option("header", True).parquet(f'{output_normalization_dir}/parquet')
    # df_filter.write.partitionBy("DATA_DATE").mode("overwrite").option("header", True).parquet(f"{target_dir}/normalization/csv")

Rows : 363270628 


+-------------------+-------------------+-------------+-------------------+-----------+--------+----+--------+-----------+----+----+-------------+-------------------------------------+---------------+-----------+-------------------------------------+-------------+----------------+---------+-----------+---------------------+-----------------+------------------------+------------------+---------+-----------+-----------+---------------+---------------+----------------------------+----------------------------+------------------+------------------+---------------+-------------+-------------+-------------+----+----+--------+-------+------+-------------------+-------------------+-------------------+-------------------+---------+-------+----------------+--------------+-------+-------------+-------------+----+-----+---------+------+------+---------------+-------------+-------------------------------------+-------------+-------------+---------------+--------------------------

In [ ]:
# this is just dummy


df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "\\N")             # treat \N as null
    .option("nanValue", "NaN")              # treat "NaN" as actual NaN
    .parquet(f'{output_normalization_dir}/parquet')
).replace([" ", "", "NULL", "nan", "NaN"], None)


# df_source = df_source.select('*').where(fn.col('POS_FIRST_ENODEB') == -1)
df_source = df_source.select('*').where(fn.col('CALL_STATUS') == 1)
# df_source = df_source.groupby('CALL_STATUS').count().orderBy('CALL_STATUS').show()
# df_pd = df_source.limit(10000).toPandas()
# df_pd.to_excel("/home/arry/Downloads/output.xlsx", index=False)


total_rows = df_source.count()

if (total_rows > 0):
    print(f"Rows : {total_rows} \n\n")
    # df_source.printSchema()
#     df_source.show(100, truncate=False)

<h3>Data Aggregation</h3>


In [4]:
import pyspark.sql.functions

df_source: Optional[DataFrame] = None
df_aggregate: Optional[DataFrame] = None


total_calls_column_name = 'TOTAL_CALLS'
total_success_calls_column_name = 'TOTAL_SUCCESS_CALLS'
total_drop_calls_column_name = 'TOTAL_DROP_CALLS'
total_block_calls_column_name = 'TOTAL_BLOCK_CALLS'
total_setup_fail_calls_column_name = 'TOTAL_SETUP_FAIL_CALLS'
total_csfb_calls_column_name = 'TOTAL_CSFB_CALLS'
# total_x2_ho_out_incomplete_calls_column_name = 'TOTAL_X2_HO_OUT_INCOMPLETE_CALLS'


src_cell_id_column_name = 'SRC_CELL_ID'
dst_cell_id_column_name = 'DST_CELL_ID'
call_status_column_name = 'CALL_STATUS'
# pos_first_enodeb_column_name = 'POS_FIRST_ENODEB'
start_enodeb_column_name = 'START_ENODEB_ID'
end_enodeb_column_name = 'END_ENODEB_ID'
avg_rsrp_column_name = 'AVG_RSRP'
avg_rsrq_column_name = 'AVG_RSRQ'
avg_ta_column_name = 'AVG_TA'
dl_throughput_column_name = 'DL_THROUPUT'
ul_throughput_column_name = 'UL_THROUPUT'
dl_volume_column_name = 'DL_VOLUME'
ul_volume_column_name = 'UL_VOLUME'
ping_pong_ho_count_column_name = 'PING_PONG_HO_CNT'


cell_avg_rsrp_column_name = 'CELL_AVG_RSRP'
cell_p10_rsrp_column_name = 'CELL_P10_RSRP'
cell_p90_rsrp_column_name = 'CELL_P90_RSRP'
cell_avg_rsrq_column_name = 'CELL_AVG_RSRQ'
cell_avg_ta_column_name = 'CELL_AVG_TA'
cell_p90_ta_column_name = 'CELL_P90_TA'
cell_total_ping_pong_ho_count_column_name = 'CELL_TOTAL_PING_PONG_HO_COUNT'
cell_avg_dl_throughput_column_name = 'CELL_AVG_DL_THROUPUT'
cell_avg_ul_throughput_column_name = 'CELL_AVG_UL_THROUPUT'
cell_total_dl_volume_column_name = 'CELL_TOTAL_DL_VOLUME'
cell_total_ul_volume_column_name = 'CELL_TOTAL_UL_VOLUME'
cell_success_rate_column_name = 'CELL_SUCCESS_RATE'
cell_drop_rate_column_name = 'CELL_DROP_RATE'
cell_block_rate_column_name = 'CELL_BLOCK_RATE'


df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "\\N")             # treat \N as null
    .option("nanValue", "NaN")              # treat "NaN" as actual NaN
    .parquet(f'{output_normalization_dir}/parquet')
).replace([" ", "", "NULL", "nan", "NaN"], None)


# CALL_TYPE:
# 21: Data Call
# 22: Signalling Call
# 23: VoLTE
# 24: SMS
# 29: Unspecified Call

# CALL_STATUS:
# 1: Normal Call
# 2: Drop Call
# 3: Block Call
# 4: Unspecified Call           # assume as unneeded junk
# 5: CS-Fallback Call
# 6: Setup Failure Call
# 7; X2 HO Out (Incomplete Correlation)


# Add new column
# ========================================================================================================================================================

# CONCAT START_CELL_ID + '_' + POS_FIRST_ENODEB

df_aggregate = (
    df_source.orderBy(start_cell_id_column_name).orderBy(start_enodeb_column_name)
)

# df_aggregate.printSchema()

# ========================================================================================================================================================


# Aggregate KPI per cell
df_aggregate = (

    # grouping by MOD_CELL_ID
    df_aggregate
    .orderBy(src_cell_id_column_name)
    .groupBy(src_cell_id_column_name)
    .agg(
        # call_status
        fn.sum(fn.when(fn.column(call_status_column_name) == 1, 1).otherwise(0)).alias(total_success_calls_column_name),
        fn.sum(fn.when(fn.column(call_status_column_name) == 2, 1).otherwise(0)).alias(total_drop_calls_column_name),
        fn.sum(fn.when(fn.column(call_status_column_name) == 3, 1).otherwise(0)).alias(total_block_calls_column_name),
        fn.sum(fn.when(fn.column(call_status_column_name) == 5, 1).otherwise(0)).alias(total_csfb_calls_column_name),
        fn.sum(fn.when(fn.column(call_status_column_name) == 6, 1).otherwise(0)).alias(total_setup_fail_calls_column_name),
        # fn.sum(fn.when(fn.column(call_status_column_name) == 7, 1).otherwise(0)).alias(total_x2_ho_out_incomplete_calls_column_name),

        # RF stats
        fn.avg(avg_rsrp_column_name).alias(cell_avg_rsrp_column_name),
        fn.expr(f'percentile_approx({avg_rsrp_column_name}, 0.1)').alias(cell_p10_rsrp_column_name),
        fn.expr(f'percentile_approx({avg_rsrp_column_name}, 0.9)').alias(cell_p90_rsrp_column_name),
        fn.avg(avg_rsrq_column_name).alias(cell_avg_rsrq_column_name),

        # TA stats
        fn.avg(avg_ta_column_name).alias(cell_avg_ta_column_name),
        fn.expr(f'percentile_approx({avg_ta_column_name}, 0.9)').alias(cell_p90_ta_column_name),

        # throughput / volume
        fn.avg(dl_throughput_column_name).alias(cell_avg_dl_throughput_column_name),
        fn.avg(ul_throughput_column_name).alias(cell_avg_ul_throughput_column_name),
        fn.sum(dl_volume_column_name).alias(cell_total_dl_volume_column_name),
        fn.sum(ul_volume_column_name).alias(cell_total_ul_volume_column_name),

        # mobility
        fn.sum(ping_pong_ho_count_column_name).alias(cell_total_ping_pong_ho_count_column_name)
    )
)


# add TOTAL_CALLS column
df_aggregate = (
    df_aggregate.withColumn(
        total_calls_column_name, (
            fn.col(total_success_calls_column_name) + fn.col(total_drop_calls_column_name) +
            fn.col(total_block_calls_column_name) + fn.col(total_csfb_calls_column_name) +
            fn.col(total_setup_fail_calls_column_name)
        )
    )
)


# filter only if the TOTAL_CALLS > 0
df_aggregate = df_aggregate.filter(fn.col(total_calls_column_name) > 0)


# add CELL_SUCCESS_RATE, CELL_DROP_RATE & CELL_BLOCK_RATE columns
df_aggregate = (
    df_aggregate
    .withColumn(cell_success_rate_column_name,
                fn.when(fn.col(total_calls_column_name) > 0, (fn.col(total_success_calls_column_name) / fn.col(total_calls_column_name)))
                .otherwise(fn.lit(None)))
    .withColumn(cell_drop_rate_column_name,
                fn.when(fn.col(total_calls_column_name) > 0, (fn.col(total_drop_calls_column_name) / fn.col(total_calls_column_name)))
                .otherwise(fn.lit(None)))
    .withColumn(cell_block_rate_column_name,
                fn.when(fn.col(total_calls_column_name) > 0, (fn.col(total_block_calls_column_name) / fn.col(total_calls_column_name)))
                .otherwise(fn.lit(None)))

)


total_rows = df_aggregate.count()

if (total_rows > 0):
    print(f"Rows : {total_rows} \n\n")
    df_aggregate.show(10, truncate=False)
    df_aggregate.write.mode("overwrite").option("header", True).parquet(f'{output_aggregation_dir}/parquet')

    df_xls = df_aggregate.orderBy(fn.col(src_cell_id_column_name)).toPandas()
    df_xls.to_excel("/home/arry/Downloads/output.xlsx", index=False)
    # df_aggregate.select('*').where(fn.col(cell_id_column_name).like('33%')).show(truncate=False)

Rows : 6203 




+-----------+-------------------+----------------+-----------------+----------------+----------------------+-------------------+-------------------+------------------+-------------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-----------+------------------+---------------------+---------------+
|SRC_CELL_ID|TOTAL_SUCCESS_CALLS|TOTAL_DROP_CALLS|TOTAL_BLOCK_CALLS|TOTAL_CSFB_CALLS|TOTAL_SETUP_FAIL_CALLS|CELL_AVG_RSRP      |CELL_P10_RSRP      |CELL_P90_RSRP     |CELL_AVG_RSRQ      |CELL_AVG_TA       |CELL_P90_TA      |CELL_AVG_DL_THROUPUT|CELL_AVG_UL_THROUPUT|CELL_TOTAL_DL_VOLUME|CELL_TOTAL_UL_VOLUME|CELL_TOTAL_PING_PONG_HO_COUNT|TOTAL_CALLS|CELL_SUCCESS_RATE |CELL_DROP_RATE       |CELL_BLOCK_RATE|
+-----------+-------------------+----------------+-----------------+----------------+----------------------+-------------------+-------------------+------------------+-----------------

In [5]:
# this is just dummy

df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .option("nullValue", "\\N")             # treat \N as null
    .option("nanValue", "NaN")              # treat "NaN" as actual NaN
    .parquet(f'{output_aggregation_dir}/parquet')
).replace([" ", "", "NULL", "nan", "NaN"], None)


df_pd = df_source.orderBy('SRC_CELL_ID').toPandas()
df_pd.to_excel("/home/arry/Downloads/output.xlsx", index=False)